In [1]:
from astropy.io import fits
from astropy.table import Table
#import matplotlib.pyplot as plt
import numpy as np
from pykoa.koa import Koa 
import os
import time
import pandas as pd
from pathlib import Path
import csv
import matplotlib.pyplot as plt


#!pip install pandas

In [2]:
df = pd.read_csv('CHEMISTS/TOI-178/HI.20210927.41592_3_01_hdr.txt.gz', compression='gzip', sep='\t') # or sep=','

dg = pd.read_csv('CHEMISTS/TOI-178/HI.20210927.41592_3_01_flux.tbl.gz', sep='\s+') # or sep=','



In [3]:
print(df.iloc[10,:])
print(df)

SIMPLE  =                    T    AMPLOC  = '6       '           / see HIRES eng...
Name: 10, dtype: object
                        SIMPLE  =                    T
0                       BITPIX  =                  -32
1    NAXIS   =                    0 / number of dat...
2    EXTEND  =                    T / FITS dataset ...
3    DATE    = '2021-09-27T11:34:28' / file creatio...
4    DETCNFID=                 1002 / mosaic detect...
..                                                 ...
560  WFIT_NRJ=                    0 / number of arc...
561  ARC_FIT1= '  6.543138052E+03  3.089290826E-02 ...
562  ARC_FIT2= ' -2.146930017E-14  4.462965199E-18 ...
563  MK_GAIN =              0.34483 / MAKEE gain us...
564                                                END

[565 rows x 1 columns]


In [4]:
# i = 0
# while i < 200:
#     print(df.iloc[i,:])
#     i = i + 1

In [5]:
# Information I need to get and where I can get them

            # "Filename",   -- flux.tbl.gz
            # "Target",  --  Folder name ///  -- hdr.txt.gz  (row 78)
            # "RA",  -- hdr.txt.gz  (row 66)
            # "Dec",  -- hdr.txt.gz  (row 62)
            # "instrument",  -- hdr.txt.gz  (row 35)
            # "SNR_General",   -- flux.tbl.gz  [:,8]
            # "SNR-Li",
            # "SNR_Ha",
            # "SNR_Ca",
            # "wavelength-low",   -- flux.tbl.gz [0,4]
            # "wavelength-high",   -- flux.tbl.gz [len -1,4]

In [6]:
print(df.iloc[35,0])   # This form works
print(df.iloc[62,0])
print(df.iloc[66,0])
print(df.iloc[78,0])


INSTRUME= 'HIRES   '           / KOA: Instrument
DEC     = '-30:27:13.4'
RA      = '00:29:12.39'
TARGNAME= 'T000178 '


In [7]:
#print(dg.iloc[:,4])

#print(dg.iloc[

SyntaxError: incomplete input (905091700.py, line 3)

In [38]:
s = df.iloc[35,0]
try:
    start = s.index("'") + 1  # Find the first single quote and move one position past it
    end = s.index("'", start) # Find the next single quote, starting search from the 'start' position
    substringS = s[start:end]   # Slice the string between the two indices
    Sval = substringS.replace(" ","")
    print(Sval)
except ValueError:
    print("Single quotes not found or not paired correctly.")

# This code works but it pops out "HIRES    " so there is a blank space. This is due to how the txt file is formatted
## Just do the remove with the space.


# fgfgf = substring.replace(" ", "")
#     print(fgfgf)

HIRES


In [36]:
print(dg.iloc[100,4])
print(len(dg))

6547.9101
4056


In [37]:
print(dg)

           |    col     |.1     row   |raw_col   |raw_row        |.2  \
0        0.0  17.39  667.61     0.0  6544.8239  549.94836  -1.000000   
1        1.0  17.40  667.60     1.0  6544.8548  549.94836  -1.000000   
2        2.0  17.41  667.59     2.0  6544.8857  549.94836  -1.000000   
3        3.0  17.42  667.58     3.0  6544.9166  549.94836  -1.000000   
4        4.0  17.43  667.57     4.0  6544.9475  549.94836  -1.000000   
...      ...    ...     ...     ...        ...        ...        ...   
4051  4051.0  66.33  618.67  4051.0  6661.3538  500.08151  26.998039   
4052  4052.0  66.35  618.65  4052.0  6661.3797  577.02893  29.389792   
4053  4053.0  66.36  618.64  4053.0  6661.4056  659.17505  33.193901   
4054  4054.0  66.38  618.62  4054.0  6661.4316  596.53107  31.744501   
4055  4055.0  66.39  618.61  4055.0  6661.4575  619.85754  32.780755   

            wave        |.3        Flux  ...  Background  |.6  Sig_to_Noise  \
0       0.000000  -1.000000     0.00000  ...         NaN

In [39]:

# Define the directory path
folder = Path('CHEMISTS/TOI-178')

# Loop through all files in the current directory only
for file in folder.iterdir():
    if file.is_file():
        print(f"File found: {file.name}")

File found: HI.20210927.41592_3_01_flux.tbl.gz
File found: HI.20210927.41592_3_01_hdr.txt.gz
File found: HI.20210927.41592_3_02_flux.tbl.gz
File found: HI.20210927.41592_3_02_hdr.txt.gz


In [85]:
Chemfolder = Path('CHEMISTS')

Starfolder = [f for f in Chemfolder.iterdir() if f.is_dir()]

for i in range(len(Starfolder)):
    
    # 1. Define your folder
    folder = Path(Starfolder[i])
    
    # 2. Convert all files into a list (filtering out subfolders)
    files = [f for f in folder.iterdir() if f.is_file()]
    
    # 3. Loop through the list, skipping by 2 each time
    for i in range(0, len(files), 2):
        # Grab a slice of two files
        pair = files[i : i + 2]
        
        if len(pair) == 2:
            file1, file2 = pair
            #print(f"Saving pair: {file1.name} and {file2.name}")
    
            dflux = pd.read_csv(file1, sep='\s+') # or sep=','
            
            dhdr = pd.read_csv(file2, compression='gzip', sep='\t') # or sep=','
    
            # Extracting the necessay information from the hdr file.
            
            tarAtt = dhdr.iloc[78,0]
            try:
                start = tarAtt.index("'") + 1  # Find the first single quote and move one position past it
                end = tarAtt.index("'", start) # Find the next single quote, starting search from the 'start' position
                substringtarATT = tarAtt[start:end]   # Slice the string between the two indices
                tar = substringtarATT.replace(" ","")
                print(tar)
            except ValueError:
                print("Single quotes not found or not paired correctly.")
    
    
            RAAtt = dhdr.iloc[66,0]
            try:
                start = RAAtt.index("'") + 1  # Find the first single quote and move one position past it
                end = RAAtt.index("'", start) # Find the next single quote, starting search from the 'start' position
                substringRAATT = RAAtt[start:end]   # Slice the string between the two indices
                RA = substringRAATT.replace(" ","")
                print(RA)
            except ValueError:
                print("Single quotes not found or not paired correctly.")
    
    
            DecAtt = dhdr.iloc[62,0]
            try:
                start = DecAtt.index("'") + 1  # Find the first single quote and move one position past it
                end = DecAtt.index("'", start) # Find the next single quote, starting search from the 'start' position
                substringDecATT = DecAtt[start:end]   # Slice the string between the two indices
                Dec = substringDecATT.replace(" ","")
                print(Dec)
            except ValueError:
                print("Single quotes not found or not paired correctly.")
    
    
            instrAtt = dhdr.iloc[35,0]
            try:
                start = instrAtt.index("'") + 1  # Find the first single quote and move one position past it
                end = instrAtt.index("'", start) # Find the next single quote, starting search from the 'start' position
                substringinstrATT = instrAtt[start:end]   # Slice the string between the two indices
                instr = substringinstrATT.replace(" ","")
                print(instr)
            except ValueError:
                print("Single quotes not found or not paired correctly.")
            
            # All values from hdr file are found successfully.
    
            # Gets the lowest and highest wavelength from the flux file
            
            wavelow = dflux.iloc[0,4]
            print(wavelow)
    
            wavehigh = dflux.iloc[len(dflux)-1,4]
            print(wavehigh)
    
            # Wavelengths found successfully
    
            # Gets SNR value around the target wavelength
    
            min_limit = [3930, 3965, 6560, 6705]
                
            max_limit = [3935, 3970, 6565, 6710]
    
            wavelength_full = dflux.iloc[:,4]
    
            SNR = 0
            count = 0
            
            for f in range(len(min_limit)):
                for k in range(len(wavelength_full)):
                    if wavelength_full[k] > min_limit[f] and wavelength_full[k] < max_limit[f]:
                        SNR = SNR + dflux.iloc[k,8]
                        count = count + 1
                        #print(SNR)
                        
    
            AVGSNR = SNR/count
            print(AVGSNR)
    
            if wavelength_full[0] > max_limit[2]:
                line = 'Lithium'
            elif wavelength_full[0] > max_limit[1]:
                line = 'H-alpha'
            else:
                line = 'Ca II H & K'
                    
            print(line)
            # SNR found successfuly, I need to find a way to get a better SNR because right now it is one of the values in the range.
                
            
            #print(i)
    
            import csv
    
            #add = 1 +(i/2)
    
            new_row = [file1, file2, tar, RA, Dec, instr, AVGSNR, wavelow, wavehigh, line ]
    
            with open("C:/Users/Aidan Moran-Bates/Downloads/ASTR502Test2.csv", 'a', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(new_row)
    
        
        else:
            # This handles the case where you have an odd number of files
            print(f"Final single file: {pair[0].name}")

KeckI
00:11:19.54
Single quotes not found or not paired correctly.
HIRESScienceMosaic:#1(B):17-7-1#2(G):17-7-6#3(R):2-2-1
6537.493
6655.3792
92.74460922085892
H-alpha
KeckI
00:11:19.54
Single quotes not found or not paired correctly.
HIRESScienceMosaic:#1(B):17-7-1#2(G):17-7-6#3(R):2-2-1
6660.8674
6780.9169
127.96511999999993
Lithium
EPIC220503236
00:51:14.00
+06:50:47.0
HIRES
3878.532
3955.8979
4.685477040231784
Ca II H & K
EPIC220503236
00:51:14.00
+06:50:47.0
HIRES
3923.1141
3999.4667
4.802365659800365
Ca II H & K
EPIC220503236
00:51:14.00
+06:50:47.0
HIRES
3968.3994
4044.0259
4.937620944117647
Ca II H & K
EPIC220503236
00:51:14.00
+06:50:47.0
HIRES
6545.4021
6662.1194
3.3395935871951243
H-alpha
T000178
00:29:12.39
-30:27:13.4
HIRES
6544.8239
6661.4575
55.5774665121951
H-alpha
T000178
00:29:12.39
-30:27:13.4
HIRES
6668.337
6787.0981
61.10892611585367
Lithium
T004638
00:51:16.90
+12:47:16.0
HIRES
3884.8046
3939.6485
5.040369439131513
Ca II H & K
T004638
00:51:16.90
+12:47:16.0
HIRES
